# TFG — Modelo 3: Sistema de Recomendación Basado en Reglas

**Objetivo:** convertir los resultados del EDA (Modelo 1) y la regresión invernal (Modelo 2B)  
en recomendaciones operativas para mejorar la experiencia del esquiador y distribuir flujos.

**Naturaleza del sistema:** herramienta de **planificación previa**, no de guía en tiempo real.  
Ayuda a decidir *cuándo ir*, *a qué zona* y *qué nivel de presión esperar*,  
no a recomendar pistas concretas dentro de la estación.

**Lógica general:** clasificación de presión en tres niveles (BAJA / MEDIA / ALTA)  
mediante un motor de reglas de tres módulos en cascada, seguido de recomendaciones  
diferenciadas por perfil de esquiador.

> **Analogía:** igual que Waze no predice exactamente el número de vehículos  
> sino que clasifica una ruta como verde/amarilla/roja y ajusta la recomendación,  
> este sistema clasifica la presión esperada en la estación y ofrece orientación accionable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('/content/graficas', exist_ok=True)

C1, C2, C3, C4 = '#2C6E8A', '#E07B39', '#5BA55B', '#C44E52'
CVERDE, CAMARILLO, CROJO = '#4CAF50', '#FF9800', '#F44336'

# ── Carga para visualizaciones de calibración ────────────────────────────────
mm  = pd.read_csv('/content/master_mensual.csv')
gt  = pd.read_csv('/content/gtrends_clean.csv')
enc = pd.read_csv('/content/encuesta_clean_tfg.csv')
mm  = mm.merge(gt[['anio','mes','gt_esquiar']], on=['anio','mes'], how='left')
mm_oc = mm[mm['ocupacion_general_pct'].notna()].copy()
inv   = mm_oc[mm_oc['mes'].isin([11,12,1,2,3,4]) & (mm_oc['pct_dias_abierta'] > 0)].copy()
print("Datos cargados.")
print(f"  Submuestra invernal (Modelo 2B): n={len(inv)} obs · {inv['estacion'].nunique()} estaciones")

---
## 1. Fundamentos empíricos de las reglas

Cada regla del sistema deriva de un resultado concreto del EDA o del Modelo 2B.  
Esta sección muestra los datos de calibración que justifican los umbrales utilizados.

In [ ]:
# R1 — Calibración: presión por mes y por zona ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
meses_lbl = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',11:'Nov',12:'Dic'}
orden_meses = [11,12,1,2,3,4]

# Panel izq: ocupación media por mes + barras de error
mes_stats = inv.groupby('mes')['ocupacion_general_pct'].agg(['mean','std']).reindex(orden_meses)
colores_m = [CROJO if m in [2,3] else CAMARILLO if m in [1,12,4] else CVERDE for m in orden_meses]
axes[0].bar([meses_lbl[m] for m in orden_meses], mes_stats['mean'],
            color=colores_m, alpha=0.85, edgecolor='white')
axes[0].errorbar([meses_lbl[m] for m in orden_meses], mes_stats['mean'],
                 yerr=mes_stats['std'], fmt='none', color='black', capsize=4)
for i, (m, v) in enumerate(zip(orden_meses, mes_stats['mean'])):
    axes[0].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=9)
axes[0].set_ylabel('Ocupación media (%)')
axes[0].set_title('Presión por mes (temporada invernal)\n'
                  'rojo=ALTA, naranja=MEDIA, verde=BAJA', fontweight='bold')
axes[0].set_ylim(0, 70)
axes[0].axhline(46, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

# Panel dch: ocupación media por zona (inv)
zona_stats = inv.groupby('zona_hotelera')['ocupacion_general_pct'].mean().sort_values(ascending=False)
zonas_alta = ['Benasque','Vall de Boí, La','Monachil']
zonas_baja = ['Lleida','Vielha e Mijaran']
colores_z = [CROJO if z in zonas_alta else CVERDE if z in zonas_baja else CAMARILLO
             for z in zona_stats.index]
axes[1].barh(zona_stats.index, zona_stats.values, color=colores_z, alpha=0.85, edgecolor='white')
for i, v in enumerate(zona_stats.values):
    axes[1].text(v+0.3, i, f'{v:.0f}%', va='center', fontsize=9)
axes[1].set_xlabel('Ocupación media invernal (%)')
axes[1].set_title('Presión estructural por zona hotelera\n'
                  '(Modelo 2B — referencia: Benasque)', fontweight='bold')
axes[1].set_xlim(0, 70)

plt.suptitle('Modelo 3 — Calibración de umbrales de presión', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M3_01_calibracion_presion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# R2 — Calibración: gt_esquiar × ocupación dentro de cada mes ───────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
GT_UMBRALES = {11:(38,41), 12:(63,66), 1:(70,76), 2:(43,65), 3:(27,34), 4:(8,14)}

for ax, m in zip(axes.flatten(), [11,12,1,2,3,4]):
    sub = inv[inv['mes']==m].copy()
    p33, p67 = GT_UMBRALES[m]
    col = sub['gt_esquiar'].apply(lambda g: CROJO if g>p67 else CAMARILLO if g>p33 else CVERDE)
    ax.scatter(sub['gt_esquiar'], sub['ocupacion_general_pct'],
               c=col, alpha=0.6, s=25, edgecolors='none')
    ax.axvline(p33, color='gray', linestyle=':', linewidth=1)
    ax.axvline(p67, color='gray', linestyle='--', linewidth=1)
    z = np.polyfit(sub['gt_esquiar'], sub['ocupacion_general_pct'], 1)
    xr = np.linspace(sub['gt_esquiar'].min(), sub['gt_esquiar'].max(), 50)
    ax.plot(xr, np.poly1d(z)(xr), color=C2, linewidth=1.5)
    r = sub['gt_esquiar'].corr(sub['ocupacion_general_pct'])
    # medias por nivel gt
    med_bajo = sub[sub['gt_esquiar']<=p33]['ocupacion_general_pct'].mean()
    med_alto = sub[sub['gt_esquiar']>p67]['ocupacion_general_pct'].mean()
    ax.set_title(f'{list({1:"Ene",2:"Feb",3:"Mar",4:"Abr",11:"Nov",12:"Dic"}.items())[[list({1:"Ene",2:"Feb",3:"Mar",4:"Abr",11:"Nov",12:"Dic"}.keys()).index(m)]][1]}  r={r:.2f}', fontweight='bold')
    ax.set_xlabel('gt_esquiar'); ax.set_ylabel('Ocupación (%)')
    ax.text(0.02, 0.96, f'gt bajo→{med_bajo:.0f}%\ngt alto→{med_alto:.0f}%',
            transform=ax.transAxes, va='top', fontsize=7.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.grid(alpha=0.2)

handles = [mpatches.Patch(color=CVERDE,label='gt BAJO'),
           mpatches.Patch(color=CAMARILLO,label='gt MEDIO'),
           mpatches.Patch(color=CROJO,label='gt ALTO')]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5,-0.02))
plt.suptitle('Calibración: gt_esquiar vs ocupación dentro de cada mes\n'
             '(cada punto = estación-año; umbrales p33/p67)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M3_02_calibracion_gt.png', dpi=150, bbox_inches='tight')
plt.show()

print("Deltas ocupación (gt alto vs gt bajo) por mes:")
for m in [12,1,2,3]:
    sub = inv[inv['mes']==m]
    p33, p67 = GT_UMBRALES[m]
    bajo = sub[sub['gt_esquiar']<=p33]['ocupacion_general_pct'].mean()
    alto = sub[sub['gt_esquiar']>p67]['ocupacion_general_pct'].mean()
    print(f"  mes {m}: gt_bajo={bajo:.1f}%  gt_alto={alto:.1f}%  delta={alto-bajo:.1f}pp")

---
## 2. Definición formal del motor de reglas

El sistema opera en **tres módulos en cascada**. Cada módulo ajusta el nivel de presión  
calculado por el módulo anterior. El resultado final es un nivel de presión: BAJA / MEDIA / ALTA.

### Umbrales calibrados sobre Modelo 2B (n=490, nov-abr, 15 estaciones)

| Mes | gt BAJO (≤p33) | gt MEDIO (p33-p67) | gt ALTO (>p67) | Presión base |
|---|---|---|---|---|
| Noviembre | ≤38 | 38-41 | >41 | BAJA (siempre) |
| Diciembre | ≤63 | 63-66 | >66 | BAJA / BAJA / MEDIA |
| Enero | ≤70 | 70-76 | >76 | MEDIA / MEDIA / ALTA |
| Febrero | ≤43 | 43-65 | >65 | MEDIA / ALTA / ALTA |
| Marzo | ≤27 | 27-34 | >34 | MEDIA / ALTA / ALTA |
| Abril | ≤8 | 8-14 | >14 | BAJA / BAJA / MEDIA |

| Zona | Presión estructural | Ajuste |
|---|---|---|
| Benasque, Vall de Boí, Monachil | Alta (52-56%) | +1 nivel |
| Jaca, Sallent de Gállego | Media (50-51%) | Sin ajuste |
| Lleida, Vielha e Mijaran | Baja (39-40%) | -1 nivel |

> Fuente umbrales: percentiles p33/p67 de gt_esquiar calculados sobre  
> la submuestra invernal (2018-2025). Presión estructural: coeficientes  
> zona del Modelo 2B (zona_Lleida β=-8.08, zona_Vielha β=-4.28, ambas p<0.001).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MOTOR DE REGLAS — DEFINICIÓN COMPLETA
# ════════════════════════════════════════════════════════════════════════════

# ── Umbrales gt calibrados sobre Modelo 2B ───────────────────────────────────
GT_UMBRALES = {
    11: (38, 41),   # Noviembre
    12: (63, 66),   # Diciembre
     1: (70, 76),   # Enero
     2: (43, 65),   # Febrero
     3: (27, 34),   # Marzo
     4: (8,  14),   # Abril
}

# ── Clasificación de zonas por presión estructural ───────────────────────────
ZONA_ALTA   = ['Benasque', 'Vall de Boí, La', 'Monachil']
ZONA_MEDIA  = ['Jaca', 'Sallent de Gállego']
ZONA_BAJA   = ['Lleida', 'Vielha e Mijaran']

# ── Módulo 1: presión base (mes × gt_esquiar) ────────────────────────────────
def modulo1_presion_base(mes: int, gt: float) -> str:
    """
    Calcula la presión base a partir del mes y el nivel de gt_esquiar.
    Justificación: mes y gt_esquiar son los dos predictores más potentes
    del Modelo 2B (betas de mes_2, mes_3 > 7; beta gt_esquiar = 8.81, p<0.001).
    """
    p33, p67 = GT_UMBRALES[mes]
    nivel_gt = 'ALTO' if gt > p67 else 'MEDIO' if gt > p33 else 'BAJO'

    if mes == 11:
        return 'BAJA'                              # Nov: siempre baja (datos escasos, apertura incipiente)
    elif mes == 12:
        return 'MEDIA' if nivel_gt == 'ALTO' else 'BAJA'
    elif mes == 1:
        return 'ALTA' if nivel_gt == 'ALTO' else 'MEDIA'
    elif mes in [2, 3]:
        return 'BAJA' if nivel_gt == 'BAJO' else 'ALTA'  # Feb/Mar: alto o muy alto
    elif mes == 4:
        return 'MEDIA' if nivel_gt == 'ALTO' else 'BAJA'
    return 'MEDIA'

# ── Módulo 2: ajuste por zona ────────────────────────────────────────────────
def modulo2_ajuste_zona(presion: str, zona: str) -> str:
    """
    Ajusta la presión base según la presión estructural de la zona.
    Justificación: coeficientes de zona en Modelo 2B son significativos
    (Lleida β=-8.08 p<0.001; Vielha β=-4.28 p<0.001; Benasque referencia alta).
    """
    escala = {'BAJA': 0, 'MEDIA': 1, 'ALTA': 2}
    inv_escala = {0: 'BAJA', 1: 'MEDIA', 2: 'ALTA'}
    v = escala[presion]
    if zona in ZONA_ALTA:
        v = min(2, v + 1)
    elif zona in ZONA_BAJA:
        v = max(0, v - 1)
    return inv_escala[v]

# ── Módulo 3: ajuste por apertura operativa ──────────────────────────────────
def modulo3_ajuste_apertura(presion: str, pct_abierta: float) -> tuple:
    """
    Ajusta la presión según la apertura operativa de la estación.
    Si pct_abierta < 0.50: reducir presión global pero emitir aviso
    de posible concentración en pistas abiertas.
    Justificación: pct_dias_abierta β=2.64 p=0.0002 en Modelo 2B.
    El aviso de concentración es inferencia apoyada en encuesta (60% colas >30min).
    """
    escala = {'BAJA': 0, 'MEDIA': 1, 'ALTA': 2}
    inv_escala = {0: 'BAJA', 1: 'MEDIA', 2: 'ALTA'}
    aviso_concentracion = pct_abierta < 0.50
    v = escala[presion]
    if pct_abierta < 0.50:
        v = max(0, v - 1)
    return inv_escala[v], aviso_concentracion

# ── Motor principal ──────────────────────────────────────────────────────────
def evaluar_presion(mes: int, gt: float, zona: str, pct_abierta: float) -> dict:
    """
    Motor de reglas completo. Devuelve nivel de presión final y metadatos.
    """
    pb  = modulo1_presion_base(mes, gt)
    pz  = modulo2_ajuste_zona(pb, zona)
    pf, aviso_conc = modulo3_ajuste_apertura(pz, pct_abierta)
    p33, p67 = GT_UMBRALES[mes]
    nivel_gt = 'ALTO' if gt > p67 else 'MEDIO' if gt > p33 else 'BAJO'
    return {
        'presion_base': pb,
        'presion_tras_zona': pz,
        'presion_final': pf,
        'nivel_gt': nivel_gt,
        'aviso_concentracion': aviso_conc,
    }

print("Motor de reglas definido correctamente.")
print("Ejemplo rápido: mes=2, gt=70, zona='Benasque', apertura=0.95")
r = evaluar_presion(2, 70, 'Benasque', 0.95)
for k, v in r.items():
    print(f"  {k}: {v}")

---
## 3. Recomendaciones por perfil de esquiador

El nivel de presión del motor de reglas se cruza con el perfil del usuario  
para generar recomendaciones diferenciadas. El nivel es **opcional**:  
si no se declara, el sistema emite recomendaciones generales.

### Tabla de recomendaciones

| Presión | Perfil | Recomendaciones principales |
|---|---|---|
| ALTA | Principiante | Cambiar fecha / llegar antes 8:30h / valorar zona baja presión |
| ALTA | Intermedio | Llegar antes 9h / evitar remontes principales 10-12h |
| ALTA | Avanzado | Optimizar horario / explorar sectores menos transitados |
| ALTA | Sin nivel | Alerta saturación / considerar alternativa de zona |
| MEDIA | Principiante | Llegar 9-9:30h / zonas verdes menos saturadas |
| MEDIA | Intermedio/Avanzado | Condiciones normales / planificación estándar |
| MEDIA | Sin nivel | Condiciones moderadas / llegada temprana recomendable |
| BAJA | Todos | Ventana favorable / condiciones cómodas esperadas |
| BAJA + aviso concentración | Todos | Presión global baja pero pistas abiertas reducidas → posibles colas locales |

> **Justificación encuesta:** valoración media cae de 3.43 (nunca colas) a 2.00 (siempre colas).  
> El 60% de encuestados espera >30 min. La hora de llegada es accionable:  
> 165 de 224 encuestados llegan antes de las 9h.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GENERADOR DE RECOMENDACIONES
# ════════════════════════════════════════════════════════════════════════════

RECOMENDACIONES = {
    # (presion_final, nivel_esquiador) → dict con semáforo, mensajes y alternativas
    ('ALTA', 'principiante'): {
        'semaforo': '🔴 PRESIÓN ALTA',
        'alerta': 'Se espera alta afluencia. Condiciones no ideales para principiantes.',
        'acciones': [
            'Llegar antes de las 8:30h para evitar las colas en los remontes de acceso',
            'Priorizar pistas verdes y azules en sectores alejados del acceso principal',
            'Considerar cambiar la fecha a entre semana si es posible',
            'Valorar estaciones en zona Lleida o Vielha e Mijaran como alternativa',
        ],
        'consejo_zona': 'Zonas con menor presión estructural: Lleida (La Molina, Masella) o Vielha e Mijaran (Baqueira Beret sector Beret)',
    },
    ('ALTA', 'intermedio'): {
        'semaforo': '🔴 PRESIÓN ALTA',
        'alerta': 'Afluencia alta esperada. Planifica bien el día.',
        'acciones': [
            'Llegar antes de las 9:00h',
            'Evitar los remontes principales entre las 10:00 y las 12:00h (hora punta)',
            'Explorar sectores secundarios o de mayor altitud con menos tráfico',
        ],
        'consejo_zona': 'Si la zona tiene alta presión estructural, valorar alternativa en Lleida o Vielha',
    },
    ('ALTA', 'avanzado'): {
        'semaforo': '🔴 PRESIÓN ALTA',
        'alerta': 'Alta afluencia prevista. Con planificación, se puede disfrutar el día.',
        'acciones': [
            'Aprovechar las primeras horas (8:00-10:00h) cuando los remontes están menos cargados',
            'Explorar sectores fuera del circuito principal de la estación',
            'Valorar salida temprana o continuar en horario de tarde (a partir de 14h)',
        ],
        'consejo_zona': None,
    },
    ('ALTA', None): {
        'semaforo': '🔴 PRESIÓN ALTA',
        'alerta': 'Se espera alta afluencia en la zona seleccionada.',
        'acciones': [
            'Llegada temprana recomendada (antes de las 9h)',
            'Considerar alternativa en zona Lleida o Vielha e Mijaran',
            'Si vienes en finde de semana, el sábado suele tener más afluencia que el domingo',
        ],
        'consejo_zona': 'Alternativas de menor presión: Lleida, Vielha e Mijaran',
    },
    ('MEDIA', 'principiante'): {
        'semaforo': '🟡 PRESIÓN MEDIA',
        'alerta': 'Afluencia moderada. Buenas condiciones si se planifica la llegada.',
        'acciones': [
            'Llegar entre las 9:00 y las 9:30h',
            'Las zonas verdes estarán accesibles con colas moderadas',
            'Evitar la hora punta (11:00-13:00h) en remontes centrales',
        ],
        'consejo_zona': None,
    },
    ('MEDIA', 'intermedio'): {
        'semaforo': '🟡 PRESIÓN MEDIA',
        'alerta': 'Condiciones normales de temporada.',
        'acciones': [
            'Planificación estándar: llegada antes de las 9:30h',
            'Día aprovechable sin restricciones especiales',
        ],
        'consejo_zona': None,
    },
    ('MEDIA', 'avanzado'): {
        'semaforo': '🟡 PRESIÓN MEDIA',
        'alerta': 'Condiciones normales. Sin restricciones relevantes.',
        'acciones': ['Día sin condicionantes especiales. Disfruta la jornada.'],
        'consejo_zona': None,
    },
    ('MEDIA', None): {
        'semaforo': '🟡 PRESIÓN MEDIA',
        'alerta': 'Afluencia moderada esperada.',
        'acciones': [
            'Llegada temprana siempre recomendable (antes de las 9:30h)',
            'Condiciones habituales de temporada',
        ],
        'consejo_zona': None,
    },
    ('BAJA', 'principiante'): {
        'semaforo': '🟢 PRESIÓN BAJA',
        'alerta': 'Condiciones favorables. Momento cómodo para esquiar.',
        'acciones': [
            'Jornada tranquila prevista. Sin restricciones especiales.',
            'Hora de llegada flexible: el día estará cómodo desde apertura',
        ],
        'consejo_zona': None,
    },
    ('BAJA', 'intermedio'): {
        'semaforo': '🟢 PRESIÓN BAJA',
        'alerta': 'Baja afluencia prevista. Día para disfrutar sin prisa.',
        'acciones': ['Condiciones óptimas. Sin recomendaciones especiales.'],
        'consejo_zona': None,
    },
    ('BAJA', 'avanzado'): {
        'semaforo': '🟢 PRESIÓN BAJA',
        'alerta': 'Condiciones excelentes de afluencia.',
        'acciones': ['Día ideal para explorar toda la estación sin restricciones.'],
        'consejo_zona': None,
    },
    ('BAJA', None): {
        'semaforo': '🟢 PRESIÓN BAJA',
        'alerta': 'Baja presión esperada. Buen momento para visitar.',
        'acciones': ['Condiciones favorables. Llegada flexible.'],
        'consejo_zona': None,
    },
}

AVISO_CONCENTRACION = (
    "⚠️  La estación tiene apertura parcial (<50% de días operativos). "
    "Aunque la presión global es baja, las pistas disponibles pueden "
    "concentrar mayor afluencia. Consulta el estado de apertura antes de salir."
)

def recomendar(mes: int, gt: float, zona: str, pct_abierta: float,
               nivel_esquiador: str = None) -> dict:
    """
    Sistema de recomendación completo.
    nivel_esquiador: 'principiante' | 'intermedio' | 'avanzado' | None
    """
    evaluacion = evaluar_presion(mes, gt, zona, pct_abierta)
    pf = evaluacion['presion_final']
    key = (pf, nivel_esquiador)
    # fallback si nivel no está en tabla
    if key not in RECOMENDACIONES:
        key = (pf, None)
    rec = RECOMENDACIONES[key].copy()
    rec.update(evaluacion)
    rec['zona'] = zona
    if evaluacion['aviso_concentracion']:
        rec['acciones'] = [AVISO_CONCENTRACION] + rec['acciones']
    return rec

def imprimir_recomendacion(r: dict, caso: str = ''):
    print(f"{'═'*60}")
    if caso: print(f"  CASO: {caso}")
    print(f"  {r['semaforo']}")
    print(f"{'─'*60}")
    print(f"  Presión: base={r['presion_base']} → zona={r['presion_tras_zona']} → final={r['presion_final']}")
    print(f"  gt_esquiar nivel: {r['nivel_gt']}  |  Apertura estación: {'<50%' if r['aviso_concentracion'] else '≥50%'}")
    print(f"{'─'*60}")
    print(f"  {r['alerta']}")
    print(f"  Recomendaciones:")
    for accion in r['acciones']:
        print(f"    • {accion}")
    if r.get('consejo_zona'):
        print(f"  Zona alternativa: {r['consejo_zona']}")
    print()

print("Generador de recomendaciones definido.")

---
## 4. Simulación: 6 casos de ejemplo

Casos diseñados para cubrir los tres niveles de presión, distintos perfiles  
y situaciones reales de temporada invernal española.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SIMULACIÓN DE 6 CASOS DE EJEMPLO
# ════════════════════════════════════════════════════════════════════════════

casos = [
    {
        'caso': '1 — Cerler, febrero, fin de semana, gt alto',
        'desc': 'Familia con niño principiante. Febrero, gt=70 (alto para el mes).',
        'mes': 2, 'gt': 70, 'zona': 'Benasque', 'pct': 0.95, 'nivel': 'principiante',
    },
    {
        'caso': '2 — Formigal, marzo, interés digital alto',
        'desc': 'Esquiador intermedio. Marzo, gt=35 (alto: histórico p67=34).',
        'mes': 3, 'gt': 35, 'zona': 'Sallent de Gállego', 'pct': 0.90, 'nivel': 'intermedio',
    },
    {
        'caso': '3 — Candanchú/Astún, enero, sin nivel declarado',
        'desc': 'Usuario sin perfil. Enero con gt=76 (en umbral p67).',
        'mes': 1, 'gt': 76, 'zona': 'Jaca', 'pct': 1.00, 'nivel': None,
    },
    {
        'caso': '4 — La Molina/Masella, febrero, gt bajo, zona baja presión',
        'desc': 'Esquiador avanzado. Febrero con demanda digital baja (gt=40). Zona Lleida.',
        'mes': 2, 'gt': 40, 'zona': 'Lleida', 'pct': 1.00, 'nivel': 'avanzado',
    },
    {
        'caso': '5 — Cerler, diciembre, gt alto',
        'desc': 'Principiante. Diciembre con gt=70 (>p67=66). Temporada arrancando bien.',
        'mes': 12, 'gt': 70, 'zona': 'Benasque', 'pct': 0.85, 'nivel': 'principiante',
    },
    {
        'caso': '6 — Astún, abril, final temporada, apertura parcial',
        'desc': 'Principiante. Abril final de temporada. Solo 45% de días abiertos.',
        'mes': 4, 'gt': 14, 'zona': 'Jaca', 'pct': 0.45, 'nivel': 'principiante',
    },
]

resultados_sim = []
for c in casos:
    r = recomendar(c['mes'], c['gt'], c['zona'], c['pct'], c['nivel'])
    r['caso_id'] = c['caso']
    r['descripcion'] = c['desc']
    resultados_sim.append(r)
    imprimir_recomendacion(r, caso=f"{c['caso']}\n  {c['desc']}")

In [ ]:
# Visualización: tabla resumen de los 6 casos ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

MESES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',11:'Noviembre',12:'Diciembre'}
col_headers = ['Caso', 'Mes', 'gt', 'Zona', 'Apertura', 'Nivel', 'Presión final']
tabla_data = []
colores_fila = []
color_map = {'ALTA': '#FFCCCC', 'MEDIA': '#FFF3CC', 'BAJA': '#CCFFCC'}

for c, r in zip(casos, resultados_sim):
    tabla_data.append([
        f"Caso {c['caso'].split('—')[0].strip()}",
        MESES[c['mes']],
        str(c['gt']),
        c['zona'].replace('Sallent de Gállego','Sallent').replace('Vielha e Mijaran','Vielha'),
        f"{c['pct']:.0%}",
        c['nivel'] if c['nivel'] else '(sin declarar)',
        r['presion_final'],
    ])
    colores_fila.append(color_map[r['presion_final']])

table = ax.table(cellText=tabla_data, colLabels=col_headers,
                 cellLoc='center', loc='center', bbox=[0,0,1,1])
table.auto_set_font_size(False)
table.set_fontsize(9)
for j in range(len(col_headers)):
    table[0, j].set_facecolor('#2C6E8A')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i, color in enumerate(colores_fila):
    for j in range(len(col_headers)):
        table[i+1, j].set_facecolor(color)

ax.set_title('Modelo 3 — Resumen de casos simulados', fontweight='bold', pad=12, fontsize=12)
plt.tight_layout()
plt.savefig('/content/graficas/M3_04_tabla_casos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Diagrama del sistema

In [ ]:
# Diagrama de flujo del motor de reglas ─────────────────────────────────────
fig = plt.figure(figsize=(13, 7))
ax = fig.add_subplot(111)
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#E8F4F8', fontsize=9, bold=False):
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='#555', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold' if bold else 'normal', wrap=True,
            multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Inputs
box(ax, 2.0, 7.2, 3.2, 0.65, 'INPUTS DEL USUARIO
mes · zona · nivel_esquiador (opcional)', '#D6EAF8', bold=True)
box(ax, 7.5, 7.2, 3.2, 0.65, 'INPUTS AUTOMÁTICOS
gt_esquiar (Google Trends) · pct_dias_abierta', '#D5F5E3', bold=True)
arrow(ax, 2.0, 6.87, 2.0, 6.25)
arrow(ax, 7.5, 6.87, 7.5, 6.25)
arrow(ax, 3.0, 6.0, 4.5, 5.45)
arrow(ax, 7.0, 6.0, 5.5, 5.45)

# Módulos motor
box(ax, 5.0, 5.1, 3.2, 0.65, 'MÓDULO 1 — Presión base
mes × gt_esquiar → BAJA/MEDIA/ALTA', '#EBF5FB', bold=False)
arrow(ax, 5.0, 4.77, 5.0, 4.2)
box(ax, 5.0, 3.87, 3.2, 0.65, 'MÓDULO 2 — Ajuste zona
±1 nivel según presión estructural', '#EBF5FB', bold=False)
arrow(ax, 5.0, 3.53, 5.0, 2.97)
box(ax, 5.0, 2.63, 3.2, 0.65, 'MÓDULO 3 — Ajuste apertura
pct_dias_abierta < 50% → -1 nivel + aviso', '#EBF5FB', bold=False)
arrow(ax, 5.0, 2.30, 5.0, 1.75)

# Resultado presion
box(ax, 5.0, 1.45, 2.6, 0.55, 'NIVEL DE PRESIÓN FINAL
🟢 BAJA  🟡 MEDIA  🔴 ALTA', '#FEF9E7', bold=True, fontsize=9)
arrow(ax, 5.0, 1.17, 5.0, 0.7)

# Recomendaciones
box(ax, 2.2, 0.45, 2.4, 0.6, 'Horario
recomendado', '#FADBD8')
box(ax, 5.0, 0.45, 2.4, 0.6, 'Zona alternativa
(si aplica)', '#FADBD8')
box(ax, 7.8, 0.45, 2.4, 0.6, 'Alerta / ventana
de oportunidad', '#FADBD8')
arrow(ax, 3.7, 0.7, 3.3, 0.6)
arrow(ax, 6.3, 0.7, 6.7, 0.6)

ax.set_title('Modelo 3 — Arquitectura del sistema de recomendación',
             fontweight='bold', fontsize=12, pad=8)
plt.tight_layout()
plt.savefig('/content/graficas/M3_05_diagrama_sistema.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Clasificación de solidez de las reglas

| Regla | Módulo | Solidez | Respaldo empírico |
|---|---|---|---|
| Presión alta en feb-mar | M1 | ⭐⭐⭐ ALTA | Beta mes_2=7.24, mes_3=9.09 (p<0.001); media ocup feb=51% |
| gt_esquiar como modulador | M1 | ⭐⭐⭐ ALTA | Beta gt=8.81 (p<0.001); r intra-mes 0.40-0.54; delta +6-8pp |
| Noviembre siempre baja | M1 | ⭐⭐ MEDIA | Solo n=20; apertura incipiente; dirección clara pero datos escasos |
| Lleida/Vielha baja presión | M2 | ⭐⭐⭐ ALTA | Beta zona_Lleida=-8.08, zona_Vielha=-4.28 (ambas p<0.001) |
| Benasque alta presión | M2 | ⭐⭐⭐ ALTA | Referencia del modelo; 72% meses en ALTA; media ocup=55.9% |
| Apertura <50% → -1 nivel | M3 | ⭐⭐ MEDIA | Beta pct_dias=2.64 (p=0.0002); dirección correcta |
| Aviso concentración pistas | M3 | ⭐⭐ MEDIA | Inferencia + encuesta (60% colas >30min); no medido directamente |
| Rec. horario llegada | M4 | ⭐⭐ MEDIA | Encuesta: 74% llega antes 9h; correlación colas-valoración r=-0.37 |
| Rec. principiante + ALTA | M4 | ⭐⭐ MEDIA | Lógica operativa + encuesta; cruce nivel×colas no medido |
| Rec. avanzado + ALTA | M4 | ⭐ BAJA | Principalmente inferencia; datos de encuesta no discriminan por nivel |
| Vall de Boí alta presión | M2 | ⭐ BAJA | n=7 en submuestra; no generalizable |

**Síntesis:** el núcleo del sistema (M1 + M2 con las zonas principales) tiene respaldo empírico  
sólido. Las recomendaciones de perfil de usuario (M4) tienen respaldo parcial (encuesta + lógica)  
y deben presentarse como tal en la memoria.

---
## 7. Propuesta de redacción para la memoria

### 5.1. Descripción del sistema

El Modelo 3 constituye el componente propositivo del trabajo: un sistema de recomendación  
basado en reglas que convierte los resultados analíticos del EDA y del modelo de regresión  
invernal en orientaciones accionables para el esquiador. El sistema está diseñado como  
herramienta de planificación previa —no de guía en tiempo real dentro de la estación—  
respondiendo a la pregunta: *¿cuándo ir, a qué zona y qué nivel de presión esperar?*

La lógica central se inspira en los sistemas de información de tráfico tipo Waze o Google Maps:  
al igual que estos no predicen con exactitud el número de vehículos sino que clasifican  
el nivel de congestión y ajustan las recomendaciones en consecuencia, el sistema propuesto  
clasifica la presión esperada en la estación en tres niveles (BAJA, MEDIA, ALTA) y genera  
recomendaciones diferenciadas según el perfil del usuario.

### 5.2. Arquitectura del sistema

El motor de reglas opera en tres módulos en cascada:

**Módulo 1 (presión base)** combina el mes de la temporada y el nivel de `gt_esquiar`  
(índice de interés digital de Google Trends, escala 0-100) para determinar la presión base.  
Ambas variables son los predictores con mayor peso estandarizado en el Modelo 2B  
(beta mes_2=7.24, mes_3=9.09; beta gt_esquiar=8.81, todos p<0.001). Los umbrales  
de gt_esquiar se calibran mediante percentiles p33 y p67 calculados dentro de cada mes  
sobre la submuestra invernal 2018-2025, de forma que los niveles BAJO/MEDIO/ALTO  
sean siempre relativos al comportamiento histórico del mes correspondiente.

**Módulo 2 (ajuste por zona)** corrige el nivel de presión base según la presión estructural  
de la zona hotelera, derivada de los coeficientes de zona del Modelo 2B. Las zonas Lleida  
(β=-8.08, p<0.001) y Vielha e Mijaran (β=-4.28, p<0.001) reciben un ajuste negativo  
de un nivel; las zonas de alta presión estructural (Benasque, Monachil) reciben un ajuste  
positivo. Las zonas sin coeficiente significativo (Jaca, Sallent de Gállego) no se ajustan.

**Módulo 3 (ajuste por apertura operativa)** incorpora el porcentaje de días del mes  
en que la estación estuvo operativa (`pct_dias_abierta`), variable significativa en el  
Modelo 2B (β=2.64, p=0.0002). Cuando la apertura es inferior al 50% —situación  
característica de noviembre y final de abril—, la presión global se reduce un nivel  
pero se emite un aviso de posible concentración de afluencia en las pistas disponibles,  
inferencia apoyada en la encuesta (60% de esquiadores reporta esperas superiores a 30 minutos).

### 5.3. Limitaciones del sistema

El sistema opera a nivel de zona hotelera, no de pista o remonte específico.  
La ausencia de datos de afluencia intradiaria impide recomendar itinerarios dentro  
de la estación, función que requeriría sensores o datos de forfaits en tiempo real  
no disponibles en este trabajo. `gt_esquiar` es una variable nacional que no discrimina  
entre estaciones del mismo mes y zona; su capacidad predictiva opera a nivel temporal  
(variación interanual) pero no espacial. Las recomendaciones por perfil de esquiador  
tienen respaldo parcial: las relativas a horario de llegada y nivel de colas están  
avaladas por la encuesta, pero el cruce entre nivel de habilidad y tipo de saturación  
no está directamente medido en los datos disponibles.

### 5.4. Conexión con un prototipo de aplicación

La arquitectura del sistema es directamente implementable como interfaz de tres  
preguntas (*¿cuándo vas? · ¿a qué zona? · ¿cuál es tu nivel?*) más un panel  
de resultados con el semáforo de presión y las recomendaciones personalizadas.  
Los inputs automáticos —`gt_esquiar` mediante la API pública de Google Trends  
y `pct_dias_abierta` mediante scraping de infonieve.com— son accesibles sin coste.  
Esta viabilidad técnica, junto con el respaldo empírico de las reglas, permite  
proponer el sistema como base de un prototipo funcional dentro del marco  
de una aplicación de movilidad inteligente orientada al esquí.

---
## Compresión y descarga

In [ ]:
import zipfile
zip_path = '/content/graficas_modelo3.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir('/content/graficas'):
        if fname.startswith('M3'):
            zf.write(f'/content/graficas/{fname}', fname)
print(f"ZIP generado: {zip_path}")
from google.colab import files
files.download(zip_path)